# Task Planning Agent
이 섹션에서는 위에서 실행한 **LangChain Multi-Agent System (RAG + Multimodal)** 코드의 각 단계별 동작 원리를 설명합니다.

# 3일차 프로젝트 개요

## 프로젝트 주제

Qwen3-VL (GGUF 양자화 모델)과 LangChain Multi-Agent System을 결합하여, 사용자의 멀티모달 요구사항(이미지 분석 + 문서 검색)을 처리하고 개발 과업을 자동 설계하는 **지능형 태스크 플래닝 에이전트** 구축

---

## 수행 Task 시나리오

사용자가 UI 목업 이미지와 함께 특정 기능(예: 소셜 로그인) 개발을 요청하면:

1. **Vision Agent** — 이미지를 분석
2. **Planning Agent** — RAG(ChromaDB) 기반으로 PRD 문서를 검색하여 구체적인 개발 태스크(UI, API, DB)를 도출
3. **Supervisor** — 전체 개발 방향과 진척 상황을 브리핑하는 공정을 구현

---

## 데이터셋 개요

| 유형 | 설명 |
|------|------|
| **시각 데이터** | UI 목업 이미지 (시각 정보 처리용) |
| **텍스트 데이터** | 애플리케이션 제품 요구사항 정의서(PRD) PDF |


### 🛠️ 1. 환경 설정 및 모델 준비 (Step 1 ~ 5)
- **Step 1 (라이브러리 설치)**: `langchain`, `langchain-openai` 등 에이전트 구축을 위한 핵심 패키지를 설치합니다.
- **Step 2 (환경 설정)**: Qwen3-VL GGUF 모델 경로, llama-server 바이너리 경로, 서버 설정을 초기화합니다.
- **Step 3 (llama-server 실행)**: GGUF 양자화된 Qwen3-VL 모델을 llama-server로 로드하여 OpenAI 호환 API 서버를 시작합니다.
- **Step 4 (Vision Agent 도구 정의)**: llama-server에 연결된 `ChatOpenAI`를 사용하여 Qwen3-VL 기반 Vision Agent 도구를 정의합니다.
- **Step 5 (API 키 설정)**: Supervisor Agent가 사용할 OpenAI 호환 모델(gpt-4o-mini)의 API 키 및 엔드포인트를 설정합니다.

In [ ]:
# # ========================================================
# # [Step 1] 기본 라이브러리 설치
# # ========================================================
# !pip install -q "langchain>=1.0" langchain-openai langchain-mcp-adapters \
#   langchain-text-splitters \
#   "langchain-chroma>=0.1.2" chromadb \
#   docling-mcp \
#   nest_asyncio httpx pillow requests

# print("설치가 완료되었습니다.")

In [ ]:
# ========================================================
# [Step 2] 환경 설정 및 글로벌 변수 정의
# ========================================================
import os
import requests
import base64
import mimetypes
from PIL import Image

# ─── GGUF 모델 경로 ───
GGUF_MODEL_DIR = "/home/user/MultiAgents/Day_2/Ex/qwen3-vision-finetune-edu/gguf_models"

Q4_GGUF_PATH = os.path.join(GGUF_MODEL_DIR, "qwen3-vl-q4_k_m.gguf")
MMPROJ_GGUF_PATH = os.path.join(GGUF_MODEL_DIR, "qwen3-vl-mmproj-f16.gguf")

# ─── 서버 설정 ───
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8181
LLAMA_SERVER_BIN = "/home/user/MultiAgents/Day_2/Ex/llama.cpp/build/bin/llama-server"

# ─── 비전(멀티모달) 사용 여부 ───
USE_VISION = True  # False로 변경하면 텍스트 전용 모드

# ─── 확인 ───
print("📁 설정 확인:")
print(f"  GGUF 모델:     {Q4_GGUF_PATH}  {'✅' if os.path.exists(Q4_GGUF_PATH) else '❌ 파일 없음'}")
print(f"  비전 모델:     {MMPROJ_GGUF_PATH}  {'✅' if os.path.exists(MMPROJ_GGUF_PATH) else '⚠️ 파일 없음 (텍스트 전용)'}")
print(f"  llama-server:  {LLAMA_SERVER_BIN}  {'✅' if os.path.exists(LLAMA_SERVER_BIN) else '❌ 바이너리 없음'}")
print(f"  서버 주소:     http://{SERVER_HOST}:{SERVER_PORT}")
print(f"  비전 사용:     {USE_VISION}")

In [ ]:
# ========================================================
# [Step 3] llama-server 실행 (Qwen3-VL GGUF 모델 로드)
# ========================================================
import subprocess
import time

def start_llama_server():
    """llama-server를 OpenAI 호환 API 서버로 실행합니다."""

    # 모델 파일 존재 여부 확인
    if not os.path.exists(Q4_GGUF_PATH):
        raise FileNotFoundError(
            f"❌ GGUF 모델 파일을 찾을 수 없습니다: {Q4_GGUF_PATH}\n"
            f"   Q4_GGUF_PATH를 실제 모델 경로로 수정하세요."
        )

    if not os.path.exists(LLAMA_SERVER_BIN):
        raise FileNotFoundError(
            f"❌ llama-server를 찾을 수 없습니다: {LLAMA_SERVER_BIN}\n"
            f"   llama.cpp 빌드를 먼저 실행하세요."
        )

    # 이전 프로세스 정리
    os.system("pkill -f llama-server 2>/dev/null || true")
    time.sleep(2)

    # 서버 시작 명령 구성
    cmd = [
        LLAMA_SERVER_BIN,
        "-m", Q4_GGUF_PATH,
        "--host", SERVER_HOST,
        "--port", str(SERVER_PORT),
        "--jinja",              # tool calling 지원
        "-fa", "auto",          # Flash Attention
        "-ngl", "99",           # GPU 전체 오프로드
        "-c", "4096",           # 컨텍스트 길이
        "--temp", "0.1",
    ]

    # 비전 모델 사용 시 mmproj 추가
    if USE_VISION and os.path.exists(MMPROJ_GGUF_PATH):
        cmd.extend(["--mmproj", MMPROJ_GGUF_PATH])
        print(f"🖼️  비전 모델 활성화: {os.path.basename(MMPROJ_GGUF_PATH)}")

    print(f"🚀 llama-server를 시작합니다...")
    print(f"   텍스트 모델: {os.path.basename(Q4_GGUF_PATH)}")

    server_process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    # 헬스체크 (최대 120초 대기)
    print("⏳ 서버 준비 대기 중 (모델 로드에 시간이 걸릴 수 있습니다)...")
    for i in range(120):
        try:
            resp = requests.get(
                f"http://{SERVER_HOST}:{SERVER_PORT}/health", timeout=2
            )
            if resp.status_code == 200:
                print(f"✅ llama-server 준비 완료! (http://{SERVER_HOST}:{SERVER_PORT})")
                return server_process
        except requests.ConnectionError:
            pass
        time.sleep(1)

    # 실패 시 로그 출력
    stderr_output = server_process.stderr.read().decode()
    server_process.terminate()
    raise RuntimeError(f"❌ 서버 시작 실패.\n로그:\n{stderr_output}")


# 서버 시작
server_process = start_llama_server()

# 서버 속성 확인
try:
    props = requests.get(f"http://{SERVER_HOST}:{SERVER_PORT}/props").json()
    print(f"📋 Chat template: {props.get('chat_template', 'N/A')[:80]}...")
except Exception as e:
    print(f"⚠️ 속성 확인 실패: {e}")

In [ ]:
# ========================================================
# [Step 4] Vision Agent 도구 정의 (Qwen3-VL 기반)
# ========================================================
import os, requests, base64, mimetypes
from PIL import Image

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 1) llama-server에 연결된 Qwen3-VL LLM 인스턴스
vlm_llm = ChatOpenAI(
    model="qwen3-vl-q4",
    base_url=f"http://{SERVER_HOST}:{SERVER_PORT}/v1",
    api_key="not-needed",
    temperature=0.1,
    max_tokens=512,
)

def _image_to_data_uri(image_path_or_url: str) -> str:
    """이미지 URL 또는 로컬 파일 경로를 data URI로 변환합니다."""
    if image_path_or_url.startswith("http"):
        return image_path_or_url
    mime = mimetypes.guess_type(image_path_or_url)[0] or "image/png"
    with open(image_path_or_url, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return f"data:{mime};base64,{b64}"

# 2) Tool: 이미지+텍스트 → 텍스트 생성
@tool
def qwen3vl_answer(image: str, question: str) -> str:
    """Qwen3-VL 모델로 이미지+질문을 받아 답변 텍스트를 생성."""
    image_url = _image_to_data_uri(image)

    message = HumanMessage(
        content=[
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": question},
        ]
    )

    response = vlm_llm.invoke([message])
    return response.content.strip()

print("✅ Qwen3-VL Vision Agent 도구 정의 완료!")

In [ ]:
# ========================================================
# [Step 5] API 키 설정 (OpenAI)
# ========================================================
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv("/home/user/MultiAgents/.env")

# 모델 정의
model = init_chat_model(
    "gpt-4o-mini",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # 핵심: OpenAI SDK의 base_url과 동일 개념
    temperature=0,    
)

### 📄 2. 문서 처리 파이프라인: Docling MCP → RAG (Step 6)

이 섹션에서는 PDF 문서를 마크다운으로 변환하고, 이를 벡터 저장소에 인덱싱하는 **문서 처리 파이프라인**을 구축합니다.

```
PDF 문서 ──→ Markdown 텍스트 ──→ [청크 분할] ──→ [ChromaDB 벡터 저장소] ──→ RAG Retriever
```

In [ ]:
# ========================================================
# [Step 6] RAG 벡터 저장소 구축 (ChromaDB)
# ========================================================
import shutil
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from docling.document_converter import DocumentConverter

# 변환할 PDF 파일 경로입니다.
source = # UI_data의 PRD 파일 경로

# Docling 변환기를 생성합니다.
converter = DocumentConverter()

# PDF를 Docling 문서 객체로 변환합니다.
result = # Docling 문서 컨버터

# 변환된 문서를 마크다운 문자열로 추출합니다.
markdown_text = # result 결과를 markdown 변환

# 변환 결과를 확인합니다.
print(markdown_text)

# Chroma 저장 경로를 지정합니다.
# [경고] 같은 경로에 같은 이름의 저장소를 만들 수 없습니다. 
# 저장소 이름을 변경하거나 프로세스를 초기화해야합니다.
PERSIST_DIR = "/tmp/chroma_app_prd_db"

# 기존 저장소가 있다면 삭제합니다.
shutil.rmtree(PERSIST_DIR, ignore_errors=True)

# 텍스트를 청크 단위로 분할하는 설정입니다.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=# TODO,
    chunk_overlap=# TODO
)

# 마크다운 문자열을 청크로 분할합니다.
chunks = splitter.split_text(markdown_text)

# 임베딩 모델을 초기화합니다.
emb = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],
)

# Chroma 벡터 저장소를 생성합니다.
vector_store = Chroma(
    collection_name="app_prd_docs",
    embedding_function=# TODO,
    persist_directory=PERSIST_DIR,
)

# 분할된 텍스트 청크들을 벡터 저장소에 추가합니다.
vector_store.add_texts(texts=# TODO)

# Retriever를 생성합니다.
retriever = vector_store.as_retriever(search_kwargs={"k": # TODO})

# 저장된 청크 개수를 출력합니다.
print(f"✅ Indexed PRD chunks: {len(chunks)}")

### 🧠 3. 멀티 에이전트 오케스트레이션
- **Step 7 (유틸리티 함수)**: 에이전트 결과에서 마지막 메시지를 추출하는 헬퍼 함수를 정의합니다.
- **Supervisor Agent 구성**:
  - **Planning Agent (기획자)**: RAG를 통해 요구사항을 분석하고 할 일(Task)을 정의합니다.
  - **Dev Agent (개발자)**: 기술 스택과 구현 방안을 설계합니다.
  - **Tracker Agent (관리자)**: 진행 상황을 모니터링합니다.
  - **Vision Analyst (디자이너)**: Qwen3-VL을 통해 UI 이미지를 분석합니다.
  - **Supervisor (PM)**: 이 모든 에이전트를 관리하며 사용자 질문에 맞춰 적절한 에이전트를 호출합니다.
- **(실행 및 분석)**: 사용자의 복합 명령(이미지 분석 + 기획 + 개발 방향)을 수행하고, 에이전트 간의 대화 로그(Trace)를 분석합니다.

In [ ]:
# ========================================================
# [Step 7] 유틸리티 함수 정의
# ========================================================

def _last_content(agent_result: dict) -> str:
    """에이전트 결과에서 마지막 메시지의 content를 추출.
    AIMessage는 Pydantic 모델이므로 .get()이 아닌 .content 사용.
    """
    last = agent_result["messages"][-1]
    # AIMessage, HumanMessage 등 LangChain 메시지 객체
    if hasattr(last, "content"):
        return last.content
    # dict 형태 (raw message)
    if isinstance(last, dict):
        return last.get("content", "")
    return str(last)

In [ ]:
# ========================================================
# Multi-Agent Supervisor 구성
# ========================================================
from langchain.agents import create_agent
from langchain.tools import tool

# ── Sub-Agent 4: UI/UX 비전 분석가 (Qwen3-VL 기반) ──
@tool
def ask_vision_analyst(image_url: str, request: str) -> str:
    # TODO 비전 분석 툴 제작 """ """
    try:
        # Qwen3-VL의 성능을 끌어올리기 위한 프롬프트 엔지니어링
        enhanced_question = (
            f"# TODO {request}"
            )
        print(enhanced_question)
        result = qwen3vl_answer.invoke({"image": image_url, "question": # TODO})
        print(result)

        if len(result.split()) < 3:  # 결과가 너무 짧을 경우 재시도
            result = qwen3vl_answer.invoke({"image": image_url, "question": f"Detect and describe all relevant UI components in this screenshot: {request}"})

        return f"[Vision Agent 분석 결과]\n{result}"
    except Exception as e:
        return f"Vision Agent Error: {str(e)}"

# ── 1) PRD 검색 툴 (ChromaDB RAG) ──────────────
@tool
def prd_search(query: str) -> str:
    # TODO PRD 툴 설명 작성 """ """
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant PRD content found."
    out = [
        f"[source] {d.metadata.get('source', 'unknown')}\n{d.page_content[:900]}"
        for d in docs
    ]
    return "\n\n---\n\n".join(out)


# ── 2) Sub-Agent 1: 기획자 (RAG 기반 Planning Agent) ──
planning_agent = create_agent(
    vlm_llm,
    tools=[prd_search],
    system_prompt=(
        # 기획자 프롬프트 작성
    ),
    name="planning_agent",
)

# ── 3) Sub-Agent 2: 개발자 ────────────────────
dev_agent = create_agent(
    vlm_llm,
    tools=[],
    system_prompt=(
        # 개발자 프롬프트 작성
    ),
    name="dev_agent",
)

# ── 4) Sub-Agent 3: 진척 관리자 ────────────────
tracker_agent = create_agent(
    vlm_llm,
    tools=[],
    system_prompt=(
        # PM (진척 관리자) 프롬프트 작성
    ),
    name="tracker_agent",
)


# ── Sub-Agent → Tool 래핑 ─────────────────────
@tool
def ask_planner(request: str) -> str:
    # 서브에이전트 구현 """ """
    result = planning_agent.invoke(
        {"messages": [{"role": "user", "content": # TODO}]}
    )
    return _last_content(result)


@tool
def ask_developer(request: str) -> str:
    # 서브에이전트 구현 """ """
    result = dev_agent.invoke(
        {"messages": [{"role": "user", "content": # TODO}]}
    )
    return _last_content(result)


@tool
def check_progress(request: str) -> str:
    # 서브에이전트 구현 """ """
    result = tracker_agent.invoke(
        {"messages": [{"role": "user", "content": # TODO}]}
    )
    return _last_content(result)


# ── Supervisor Agent (PM) 시스템 프롬프트 보강 ──
# PM이 Vision Agent에게 더 구체적인 질문을 던지도록 지시합니다.
supervisor = create_agent(
    model,
    tools=# TODO,
    system_prompt=(
        # TODO
    ),
    name="pm_supervisor",
)

In [ ]:
# ========================================================
# [Step 12] 에이전트 실행 (Query)
# ========================================================
query = (
    "이 UI 목업 이미지를 분석해주세요. "
    "이미지 경로: /home/user/MultiAgents/Day_3/Projects/UI_data/UI.jpg"
    "필요한 개발 태스크(UI, API, DB)를 도출해줘."
)

# 비동기 실행
result = await supervisor.ainvoke(
    {"messages": [{"role": "user", "content": query}]}
)

print(_last_content(result))

In [ ]:
# ========================================================
# [Step 13] 실행 결과 분석 (Trace Parsing)
# ========================================================
import json

def extract_tools_and_responses(data):
    """
    대화 데이터에서 tool_calls 내역과 그에 대응하는 도구 실행 응답(content)을
    매핑하여 리스트 형태로 반환합니다.
    """
    extracted_data = []

    # id를 키(key)로 사용하여 툴 호출 정보를 임시 저장하고 매핑하기 위한 딕셔너리
    tool_call_dict = {}

    for msg in data.get('messages', []):
        # 메시지가 딕셔너리인지 객체(LangChain Message)인지 확인
        is_dict = isinstance(msg, dict)

        # 1. 툴 호출(tool_calls) 내역 추출 (주로 AIMessage)
        tool_calls = msg.get('tool_calls', []) if is_dict else getattr(msg, 'tool_calls', [])

        for tc in tool_calls:
            tc_id = tc.get('id')
            tool_info = {
                'tool_name': tc.get('name'),
                'arguments': tc.get('args'),
                'response': None # 응답이 들어올 자리를 미리 생성
            }
            tool_call_dict[tc_id] = tool_info
            extracted_data.append(tool_info) # 순서 유지를 위해 결과 리스트에 추가

        # 2. 툴 실행 결과(response) 추출 (주로 ToolMessage)
        tool_call_id = msg.get('tool_call_id') if is_dict else getattr(msg, 'tool_call_id', None)
        content = msg.get('content') if is_dict else getattr(msg, 'content', None)

        # tool_call_id가 존재하고, 앞서 저장해둔 툴 호출 내역에 해당 id가 있다면 응답 내용 업데이트
        if tool_call_id and tool_call_id in tool_call_dict:
            tool_call_dict[tool_call_id]['response'] = content

    return extracted_data

# ---------------------------------------------------------
# [사용 예시]
# 'data' 변수에 질문하신 데이터 전체가 담겨있다고 가정합니다.
# ---------------------------------------------------------

tool_calls_data = extract_tools_and_responses(result)

# 보기 좋게 JSON 포맷으로 출력
print(json.dumps(tool_calls_data, indent=2, ensure_ascii=False))